# Step 1: APTOS 2019 Retinal Fundus Preprocessing & Stratified Split

Prepares the **Kaggle APTOS 2019 Blindness Detection** dataset (5-Stage ICDR Grading: `0: No DR`, `1: Mild NPDR`, `2: Moderate NPDR`, `3: Severe NPDR`, `4: Proliferative DR`), previews **Ben Graham retinal illumination normalization**, and builds a leak-safe `data/metadata.csv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.preprocess import generate_aptos2019_metadata, apply_ben_graham_preprocessing, DR_CLASS_NAMES
print('Project Root:', ROOT_DIR)

### 1. Optional: Download APTOS 2019 via Kaggle CLI (Google Colab / Local)
Upload your `kaggle.json` API token first, then uncomment and run the commands below.

In [ ]:
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c aptos2019-blindness-detection -p data/aptos2019
# !unzip -q data/aptos2019/aptos2019-blindness-detection.zip -d data/aptos2019

### 2. Generate Stratified `data/metadata.csv` (80% Train / 10% Val / 10% Test)

In [ ]:
APTOS_DIR = str(ROOT_DIR / 'data' / 'aptos2019')  # Or '/kaggle/input/aptos2019-blindness-detection'
generate_aptos2019_metadata(aptos_dir=APTOS_DIR, val_size=0.10, test_size=0.10, seed=42)

### 3. Visual Verification: Raw Fundus vs. Ben Graham Retinal Normalization

In [ ]:
meta_path = ROOT_DIR / 'data' / 'metadata.csv'
if meta_path.exists():
    df = pd.read_csv(meta_path)
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for grade in range(5):
        sample = df[df['label'] == grade].iloc[0]
        img_path = Path(APTOS_DIR) / sample['image_path']
        raw = Image.open(img_path).convert('RGB')
        bg = apply_ben_graham_preprocessing(raw, img_size=224)
        axes[0, grade].imshow(raw.resize((224, 224)))
        axes[0, grade].set_title(f'Raw | Grade {grade}: {DR_CLASS_NAMES[grade]}', fontsize=9)
        axes[0, grade].axis('off')
        axes[1, grade].imshow(bg)
        axes[1, grade].set_title('Ben Graham Normalized', fontsize=9)
        axes[1, grade].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Download APTOS 2019 and run Step 2 first.')